# greenload-forecast: a walkthrough

This notebook walks through the pipeline end to end: generate synthetic data, look at it, roll a trained GRU forecaster forward in closed loop, and feed its forecasts into the MPC scheduler.

It assumes `make train` has already been run (so `results/checkpoints/` exists) -- this notebook is for *looking at* the pipeline, not for the real training run itself (see `src/greenload/train.py` and `src/greenload/evaluate.py` for that, and the README for headline results).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from greenload.config import DEFAULT
from greenload.data.generate_synthetic_data import generate_raw_series
from greenload.data.dataset import build_all_splits
from greenload.evaluate import load_model, closed_loop_rollout
from greenload.scheduler import mpc
from pathlib import Path

cfg = DEFAULT
checkpoints_dir = Path("../results/checkpoints")

## 1. Synthetic data

No real trace data is used anywhere in this repo. `generate_raw_series` procedurally builds hourly workload / price / water-WUE / carbon-intensity traces for 10 synthetic datacenter locations: daily + weekly seasonality, AR(1)-correlated noise, and genuinely distinct per-location baselines (drawn from location-indexed RNG substreams).

In [ ]:
raw = generate_raw_series(cfg)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(raw["workload"][:24 * 7])
axes[0].set_title("workload, one synthetic week")
axes[1].plot(raw["carbon"][:24 * 7, :3])
axes[1].set_title("carbon intensity, 3 locations, one week")
fig.tight_layout()

## 2. Chronological sliding windows

`build_all_splits` splits each signal into train/val/test *by time* (not by shuffling first) and z-score-normalizes using train-split statistics only -- see `src/greenload/data/dataset.py` for why both of those matter.

In [ ]:
all_splits = build_all_splits(raw, cfg)
for signal, splits in all_splits.items():
    print(f"{signal:9s} train={splits.train.num_windows:3d} val={splits.val.num_windows:3d} test={splits.test.num_windows:3d}")

## 3. Closed-loop forecast rollout

This is the real inference path (`target=None`): the model feeds its own prediction back in at every step, for the full 24-hour horizon. Compare the pre-fix (pure teacher forcing) and fixed (scheduled sampling) workload forecasters against ground truth on the first test window.

In [ ]:
signal = "workload"
test_split = all_splits[signal].test
scaler = all_splits[signal].scaler

actual = scaler.denormalize(test_split.target.numpy())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(actual[0, 0], label="actual", linewidth=2)
for mode, ckpt in (("pre-fix", "baseline"), ("fixed", "fixed")):
    model = load_model(signal, "gru_baseline" if ckpt == "baseline" else "gru_fixed", cfg, checkpoints_dir, "cpu")
    pred = scaler.denormalize(closed_loop_rollout(model, test_split.source, "cpu").numpy())
    ax.plot(pred[0, 0], "--", label=mode)
ax.set_title("24h closed-loop workload rollout, first test window")
ax.set_xlabel("hour")
ax.legend()
ax.grid(alpha=0.3)

## 4. Downstream: receding-horizon MPC scheduling

Forecasts alone aren't the point -- they feed a convex MPC scheduler (`greenload.scheduler.mpc`) that decides how much workload to route to each of the 10 locations each hour, minimizing price + water-fairness + carbon-fairness cost. Here we run the full receding-horizon loop with the fixed GRU's forecasts and compare against the perfect-foresight offline oracle.

In [ ]:
def denorm_source(name):
    s = all_splits[name]
    return s.scaler.denormalize(s.test.source.numpy().reshape(s.test.num_windows, s.test.num_loc))

true_price, true_water, true_carbon = denorm_source("price").T, denorm_source("water").T, denorm_source("carbon").T
true_workload = denorm_source("workload").T

predicted = {}
for name in ("price", "water", "carbon", "workload"):
    s = all_splits[name]
    model = load_model(name, "gru_fixed", cfg, checkpoints_dir, "cpu")
    pred = s.scaler.denormalize(closed_loop_rollout(model, s.test.source, "cpu").numpy())
    pred = np.clip(pred, 0, None)
    predicted[name] = pred.reshape(s.test.num_windows, s.test.num_loc, cfg.window_size)

allocation = mpc.run_receding_horizon(
    predicted["price"], predicted["water"], predicted["carbon"], predicted["workload"],
    true_price, true_water, true_carbon, true_workload,
    l1=cfg.l1_water, l2=cfg.l2_carbon, max_cap=cfg.max_cap,
)
gru_cost = mpc.evaluate_allocation(allocation, true_price, true_water, true_carbon, cfg.l1_water, cfg.l2_carbon)

_, oracle_allocation = mpc.offline_oracle(true_price, true_water, true_carbon, true_workload, cfg.l1_water, cfg.l2_carbon, cfg.max_cap)
oracle_cost = mpc.evaluate_allocation(oracle_allocation, true_price, true_water, true_carbon, cfg.l1_water, cfg.l2_carbon)

print("GRU-MPC (fixed):", gru_cost)
print("Offline oracle:  ", oracle_cost)

## 5. Full results

See `results/metrics.json` (single source of truth, written by `python -m greenload.evaluate`) and `results/figures/` for the full naive-vs-pre-fix-vs-fixed-vs-oracle comparison and the exposure-bias before/after error-by-horizon-step plot -- and the README for the narrative writeup.